
# CheckThat 2026 Task 3 — Stage 1 API Source Usability Triage

This notebook implements **Stage 0 + Stage 1** of the API-first pipeline:

- **Stage 0:** load the dataset split, flatten local scraped evidence JSON files, and create source-level records.
- **Stage 1:** call Vertex Gemini to decide whether each scraped source has usable readable text.

For this first run, the notebook limits the run to **10 claim IDs** and guarantees that these IDs are included:

```python
30349, 30350, 30352, 30353, 30357, 30358
```

Stage 1 does **not** judge claim relevance or truth. It only decides:

> Can this source be meaningfully inspected by later stages?


In [1]:

# Colab setup: install the Google GenAI SDK
!pip install -q -U google-genai pandas tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 1.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 791.9/791.9 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.6/245.6 kB 12.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.52.0 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
google-cloud-aiplatform 1.148.1 requires google-genai<2.0.0,>=1.66.0; python_version >= "3.10", but you have google-genai 2.0.0 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 whic

In [2]:

# Authenticate to Google Cloud / Vertex AI
from google.colab import auth
auth.authenticate_user()


In [3]:

import os
import re
import json
import time
import tarfile
import zipfile
import random
import shutil
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from tqdm.auto import tqdm

from google import genai
from google.genai import types



## 1. Configuration

Edit only this section if your dataset path or Google Cloud project differs.

Expected dataset layout after extraction:

```text
WatClaimCheck/
  train.json
  valid.json
  test.json
  articles/
```


In [4]:

# -----------------------------
# Google Cloud / Vertex settings
# -----------------------------
PROJECT_ID = "clef-checkthat"
LOCATION = "global"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
)

# -----------------------------
# Stage 1 model settings
# -----------------------------
STAGE1_MODEL = "gemini-2.5-flash"
STAGE1_TEMPERATURE = 0.0
MAX_RETRIES = 5
MAX_WORKERS = 3
STAGE1_BATCH_SIZE = 4
SAVE_AFTER_EVERY_COMPLETED_BATCHES = 10

# -----------------------------
# Dataset / run settings
# -----------------------------
from google.colab import drive
drive.mount("/content/drive")

# Your dataset zip/tar file is in:
# My Drive / CheckThat_Task3_Dataset / WatClaimCheck.tar.gz
DATASET_ARCHIVE_PATH = Path("/content/drive/MyDrive/CheckThat_Task3_Dataset/test_dataset.zip")

# The notebook will extract it here inside Colab runtime
EXTRACT_ROOT = Path("/content/data")

# Keep this as None because we are extracting from the tar.gz file
DATA_ROOT_OVERRIDE = None

# Choose split. For the competition run, this will usually be "test".
SPLIT = "test"

REQUIRED_TARGET_IDS = []
N_TARGET_IDS = None
EXTRA_TARGET_IDS = []

# To control API cost, pass a packed source text to Stage 1.
# This is character-based, not token-based, but is simple and reliable for Colab.
MAX_SOURCE_CHARS_FOR_STAGE1 = 60_000
HEAD_CHARS = 45_000
TAIL_CHARS = 15_000

RUN_NAME = "test_dataset"
OUTPUT_ROOT = Path("/content/outputs")
STAGE0_OUT = OUTPUT_ROOT / "stage0" / f"flattened_sources_{RUN_NAME}.json"
STAGE1_OUT = OUTPUT_ROOT / "stage1" / f"stage1_api_triage_{RUN_NAME}.json"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
STAGE0_OUT.parent.mkdir(parents=True, exist_ok=True)
STAGE1_OUT.parent.mkdir(parents=True, exist_ok=True)

print("Configured Stage 1 run")
print("Project:", PROJECT_ID)
print("Location:", LOCATION)
print("Model:", STAGE1_MODEL)
print("Split:", SPLIT)
print("Required IDs:", REQUIRED_TARGET_IDS)
print("Output:", STAGE1_OUT)


Mounted at /content/drive
Configured Stage 1 run
Project: clef-checkthat
Location: global
Model: gemini-2.5-flash
Split: test
Required IDs: []
Output: /content/outputs/stage1/stage1_api_triage_test_dataset.json


In [5]:

# Optional smoke test: verify Vertex Gemini is reachable.
response = client.models.generate_content(
    model=STAGE1_MODEL,
    contents="Say only: Vertex Gemini test successful.",
    config=types.GenerateContentConfig(
        temperature=0.0,
        max_output_tokens=32,
    ),
)
print(response.text)


Vertex Gemini test successful.


In [6]:
from pathlib import Path
import shutil

DATASET_TAG = "test_dataset"
DRIVE_BASE = Path("/content/drive/MyDrive/CheckThat_Task3_Dataset")

paths_to_delete = [
    # Local Stage 0/1 outputs
    Path("/content/outputs/stage0") / f"flattened_sources_{DATASET_TAG}.json",
    Path("/content/outputs/stage1") / f"stage1_api_triage_{DATASET_TAG}.json",

    # Drive Stage 0/1 outputs
    DRIVE_BASE / "outputs/stage0" / f"flattened_sources_{DATASET_TAG}.json",
    DRIVE_BASE / "outputs/stage1" / f"stage1_api_triage_{DATASET_TAG}.json",

    # Optional downstream outputs for same dataset, so later stages also rerun cleanly
    DRIVE_BASE / "outputs/stage2" / f"stage2_extract_{DATASET_TAG}.json",
    DRIVE_BASE / "outputs/stage3" / f"stage3_packets_{DATASET_TAG}.json",
    DRIVE_BASE / "outputs/stage4" / f"stage4_articles_{DATASET_TAG}.json",
    DRIVE_BASE / "outputs/submission" / f"prediction_{DATASET_TAG}.json",
    DRIVE_BASE / "outputs/submission" / "prediction.json",
]

print("Deleting pipeline outputs for:", DATASET_TAG)
print()

for p in paths_to_delete:
    if p.exists():
        p.unlink()
        print("Deleted:", p)
    else:
        print("Not found:", p)

print("\nDone. Now run the notebook from the top.")

Deleting pipeline outputs for: test_dataset

Not found: /content/outputs/stage0/flattened_sources_test_dataset.json
Not found: /content/outputs/stage1/stage1_api_triage_test_dataset.json
Not found: /content/drive/MyDrive/CheckThat_Task3_Dataset/outputs/stage0/flattened_sources_test_dataset.json
Not found: /content/drive/MyDrive/CheckThat_Task3_Dataset/outputs/stage1/stage1_api_triage_test_dataset.json
Not found: /content/drive/MyDrive/CheckThat_Task3_Dataset/outputs/stage2/stage2_extract_test_dataset.json
Not found: /content/drive/MyDrive/CheckThat_Task3_Dataset/outputs/stage3/stage3_packets_test_dataset.json
Not found: /content/drive/MyDrive/CheckThat_Task3_Dataset/outputs/stage4/stage4_articles_test_dataset.json
Not found: /content/drive/MyDrive/CheckThat_Task3_Dataset/outputs/submission/prediction_test_dataset.json
Deleted: /content/drive/MyDrive/CheckThat_Task3_Dataset/outputs/submission/prediction.json

Done. Now run the notebook from the top.



## 2. Locate and load the dataset

This cell extracts `WatClaimCheck.tar.gz` if needed, then finds the directory that contains the split files and `articles/`.


In [7]:

def find_data_root(search_root: Path) -> Path:
    """Find a directory containing split JSON files and an evidence/articles folder."""
    candidates = []

    for p in [search_root] + list(search_root.rglob("*")):
        if not p.is_dir():
            continue

        has_any_split = any((p / f"{name}.json").exists() for name in ["train", "valid", "test"])
        has_evidence_folder = any((p / name).exists() for name in ["articles", "evidence_docs", "evidence"])

        if has_any_split and has_evidence_folder:
            candidates.append(p)

    if not candidates:
        raise FileNotFoundError(
            f"Could not find dataset root under {search_root}. "
            f"Expected train/valid/test JSON and articles/ or evidence_docs/."
        )

    return sorted(candidates, key=lambda x: len(x.parts))[0]

def maybe_extract_dataset() -> Path:
    if DATA_ROOT_OVERRIDE is not None:
        root = Path(DATA_ROOT_OVERRIDE)
        if not root.exists():
            raise FileNotFoundError(f"DATA_ROOT_OVERRIDE does not exist: {root}")
        return root

    if DATASET_ARCHIVE_PATH.exists():
        EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

        marker_name = ".test_dataset_extracted"
        marker = EXTRACT_ROOT / marker_name

        if not marker.exists():
            print(f"Extracting {DATASET_ARCHIVE_PATH} to {EXTRACT_ROOT} ...")

            suffixes = "".join(DATASET_ARCHIVE_PATH.suffixes).lower()

            if suffixes.endswith(".zip"):
                with zipfile.ZipFile(DATASET_ARCHIVE_PATH, "r") as z:
                    z.extractall(EXTRACT_ROOT)
            elif suffixes.endswith(".tar.gz") or suffixes.endswith(".tgz"):
                with tarfile.open(DATASET_ARCHIVE_PATH, "r:gz") as tar:
                    tar.extractall(EXTRACT_ROOT)
            else:
                raise ValueError(f"Unsupported archive format: {DATASET_ARCHIVE_PATH}")

            marker.write_text(datetime.now(timezone.utc).isoformat())

        return find_data_root(EXTRACT_ROOT)

    for root in [Path("/content/test_dataset"), Path("/content/data/test_dataset"), Path("/content/data")]:
        if root.exists():
            try:
                return find_data_root(root)
            except FileNotFoundError:
                pass

    raise FileNotFoundError(
        f"Dataset not found. Expected archive at {DATASET_ARCHIVE_PATH}, or set DATA_ROOT_OVERRIDE."
    )


def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

DATA_ROOT = maybe_extract_dataset()

if (DATA_ROOT / "articles").exists():
    ARTICLES_DIR = DATA_ROOT / "articles"
elif (DATA_ROOT / "evidence_docs").exists():
    ARTICLES_DIR = DATA_ROOT / "evidence_docs"
elif (DATA_ROOT / "evidence").exists():
    ARTICLES_DIR = DATA_ROOT / "evidence"
else:
    raise FileNotFoundError(
        f"Could not find evidence folder under {DATA_ROOT}. "
        "Expected articles/, evidence_docs/, or evidence/."
    )

SPLIT_PATH = DATA_ROOT / f"{SPLIT}.json"

if not SPLIT_PATH.exists():
    available = [p.name for p in DATA_ROOT.glob("*.json")]
    raise FileNotFoundError(f"Split file not found: {SPLIT_PATH}. Available JSON files: {available}")

examples = load_json(SPLIT_PATH)
print("DATA_ROOT:", DATA_ROOT)
print("ARTICLES_DIR:", ARTICLES_DIR)
print("SPLIT_PATH:", SPLIT_PATH)
print("Number of examples in split:", len(examples))


Extracting /content/drive/MyDrive/CheckThat_Task3_Dataset/test_dataset.zip to /content/data ...
DATA_ROOT: /content/data/test_dataset
ARTICLES_DIR: /content/data/test_dataset/evidence_docs
SPLIT_PATH: /content/data/test_dataset/test.json
Number of examples in split: 1158



## 3. Select exactly 10 target IDs

The six required IDs are checked first. The remaining slots are auto-filled from the chosen split unless `EXTRA_TARGET_IDS` is provided.


In [8]:

def get_claim_id(example) -> str:
    """Robustly read the claim id across possible dataset variants."""
    for candidate in [
        example.get("target_id"),
        example.get("id"),
        example.get("metadata", {}).get("id"),
        example.get("label", {}).get("target_id"),
    ]:
        if candidate is not None:
            return str(candidate)
    raise KeyError(f"Could not find claim id in example keys: {list(example.keys())}")


# Full dataset run: process every claim in the selected split.
TARGET_IDS = [get_claim_id(ex) for ex in examples]

print("Full dataset mode.")
print("Number of TARGET_IDS:", len(TARGET_IDS))
print("First 10 TARGET_IDS:", TARGET_IDS[:10])

Full dataset mode.
Number of TARGET_IDS: 1158
First 10 TARGET_IDS: ['30354_exclaim', '30355_exclaim', '30356_exclaim', '30360_exclaim', '30361_exclaim', '30364_exclaim', '30365_exclaim', '30366_exclaim', '30367_exclaim', '30369_exclaim']



## 4. Stage 0 — flatten local evidence JSON files

Important leakage rule: this notebook uses only `metadata.premise_articles`. It does **not** load `label.review_article` as normal evidence.


In [9]:
def get_claim_text(example) -> str:
    return (
        example.get("metadata", {}).get("claim")
        or example.get("claim")
        or example.get("text")
        or ""
    )


def get_original_rating(example):
    label = example.get("label", {}) or {}
    return label.get("original_rating", label.get("rating", ""))


def get_premise_articles(example) -> dict:
    metadata = example.get("metadata", {}) or {}
    premise = metadata.get("premise_articles") or example.get("premise_articles") or {}
    if not isinstance(premise, dict):
        raise TypeError(f"Expected premise_articles to be a dict, got {type(premise)}")
    return premise


def flatten_article_json(article_json) -> str:
    """Flatten scraped JSON content into readable text for the API."""
    if article_json is None:
        return ""
    if isinstance(article_json, list):
        parts = []
        for x in article_json:
            if x is None:
                continue
            if isinstance(x, (dict, list)):
                parts.append(json.dumps(x, ensure_ascii=False, indent=2))
            else:
                s = str(x).strip()
                if s:
                    parts.append(s)
        return "\n".join(parts)
    if isinstance(article_json, dict):
        return json.dumps(article_json, ensure_ascii=False, indent=2)
    return str(article_json)


def pack_source_text(text: str, max_chars=MAX_SOURCE_CHARS_FOR_STAGE1) -> str:
    """Keep the beginning and end of very long sources to control API cost."""
    text = text or ""
    if len(text) <= max_chars:
        return text
    head = text[:HEAD_CHARS]
    tail = text[-TAIL_CHARS:]
    omitted = len(text) - len(head) - len(tail)
    return (
        head
        + f"\n\n[... omitted {omitted} characters from the middle of this scraped source ...]\n\n"
        + tail
    )


def safe_read_article_json(article_path: Path):
    try:
        with article_path.open("r", encoding="utf-8") as f:
            return json.load(f), None
    except Exception as e:
        return None, repr(e)


def iter_file_names(value):
    """premise_articles normally maps URL -> file_name, but handle list variants just in case."""
    if isinstance(value, list):
        for x in value:
            yield str(x)
    elif value is None:
        return
    else:
        yield str(value)


def build_source_records(examples, target_ids):
    target_set = set(str(x) for x in target_ids)
    records = []

    for ex in examples:
        cid = get_claim_id(ex)
        if cid not in target_set:
            continue

        claim = get_claim_text(ex)
        original_rating = get_original_rating(ex)
        premise_articles = get_premise_articles(ex)

        for url, file_value in premise_articles.items():
            for file_name in iter_file_names(file_value):
                article_path = ARTICLES_DIR / file_name
                article_json, load_error = safe_read_article_json(article_path)
                full_text = flatten_article_json(article_json) if load_error is None else ""
                packed_text = pack_source_text(full_text)

                records.append({
                    "id": cid,
                    "claim": claim,
                    "original_rating": original_rating,
                    "url": str(url),
                    "file_name": file_name,
                    "article_path": str(article_path),
                    "source_text": packed_text,
                    "source_text_original_chars": len(full_text or ""),
                    "source_text_packed_chars": len(packed_text or ""),
                    "source_text_was_packed": len(full_text or "") > len(packed_text or ""),
                    "load_error": load_error,
                    "stage": "stage0_flattened_source",
                })

    return records

source_records = build_source_records(examples, TARGET_IDS)

print("Number of source-level records:", len(source_records))
print("Claims represented:", sorted({r["id"] for r in source_records}))

# Save Stage 0 output for debugging and resume support.
with STAGE0_OUT.open("w", encoding="utf-8") as f:
    json.dump(source_records, f, ensure_ascii=False, indent=2)

print("Saved Stage 0 flattened sources to:", STAGE0_OUT)

pd.DataFrame(source_records)[[
    "id", "file_name", "url", "source_text_original_chars", "source_text_packed_chars", "load_error"
]].head(20)


Number of source-level records: 8979
Claims represented: ['0_ambigsnopes', '100_ambigsnopes', '104_ambigsnopes', '106_ambigsnopes', '107_ambigsnopes', '109_ambigsnopes', '111_ambigsnopes', '113_ambigsnopes', '114_ambigsnopes', '116_ambigsnopes', '117_ambigsnopes', '118_ambigsnopes', '11_ambigsnopes', '120_ambigsnopes', '122_ambigsnopes', '124_ambigsnopes', '125_ambigsnopes', '127_ambigsnopes', '128_ambigsnopes', '129_ambigsnopes', '12_ambigsnopes', '130_ambigsnopes', '131_ambigsnopes', '133_ambigsnopes', '134_ambigsnopes', '135_ambigsnopes', '136_ambigsnopes', '139_ambigsnopes', '13_ambigsnopes', '140_ambigsnopes', '142_ambigsnopes', '144_ambigsnopes', '145_ambigsnopes', '146_ambigsnopes', '147_ambigsnopes', '148_ambigsnopes', '149_ambigsnopes', '14_ambigsnopes', '150_ambigsnopes', '151_ambigsnopes', '153_ambigsnopes', '154_ambigsnopes', '157_ambigsnopes', '158_ambigsnopes', '159_ambigsnopes', '161_ambigsnopes', '162_ambigsnopes', '163_ambigsnopes', '164_ambigsnopes', '165_ambigsnopes'

,id,file_name,url,source_text_original_chars,source_text_packed_chars,load_error
0,30354_exclaim,30354_1.json,https://www.youtube.com/watch?v=hfJ82VpSulk,388,388,NaN
1,30354_exclaim,30354_2.json,https://www.cnn.com/TRANSCRIPTS/1606/21/sitroo...,46067,46067,NaN
2,30354_exclaim,30354_3.json,https://www.politifact.com/truth-o-meter/state...,15203,15203,NaN
3,30354_exclaim,30354_4.json,https://www.politifact.com/truth-o-meter/state...,17635,17635,NaN
4,30354_exclaim,30354_5.json,https://money.cnn.com/2015/08/31/news/companie...,7900,7900,NaN
5,30354_exclaim,30354_6.json,https://abcnews.go.com/Politics/donald-trump-f...,8818,8818,NaN
6,30354_exclaim,30354_7.json,https://www.forbes.com/sites/debtwire/2015/08/...,10846,10846,NaN
7,30354_exclaim,30354_8.json,https://www.wsj.com/articles/hillary-clinton-t...,14776,14776,NaN
8,30354_exclaim,30354_9.json,https://www.theatlantic.com/politics/archive/2...,33936,33936,NaN
9,30354_exclaim,30354_10.json,https://www.nbcnews.com/news/us-news/trump-ban...,5069,5069,NaN



## 5. Stage 1 prompt and JSON validation

Stage 1 chooses only:

- `PROCESS`: readable content exists.
- `SKIP`: no usable readable content exists.

It must **not** judge whether the source proves, supports, or refutes the claim.


In [10]:
ALLOWED_STAGE1_DECISIONS = {"PROCESS", "SKIP"}


def source_key(source: dict) -> str:
    return f'{source["id"]}||{source["file_name"]}||{source["url"]}'


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def build_stage1_prompt(source: dict) -> str:
    return f"""You are checking whether a scraped source contains usable readable text.

You are NOT judging whether the source is relevant to the claim.
You are NOT checking whether the claim is true.
You are only deciding whether the source has meaningful readable content that a later stage can inspect.

Allowed decisions:
PROCESS = readable content exists.
SKIP = no usable readable content exists.

Choose PROCESS if the source contains any meaningful article text, post text, archive text, screenshot description, claim text, or useful page content.

Choose SKIP only if the source is empty, near-empty, login-only, access-denied, JavaScript-error-only, or menu/footer/cookie-shell-only.

Important:
- If there is readable article-like, post-like, archive-like, or claim-like content anywhere, choose PROCESS.
- Do not decide whether the source proves, supports, or refutes the claim.
- Do not copy source text into your answer.
- Do not return JSON.
- Return exactly three short lines in this format:

DECISION: PROCESS
CONFIDENCE: 0.95
REASON: readable article/post/archive/page content exists

Input:
Claim ID: {source["id"]}
Claim: {source["claim"]}
URL: {source["url"]}
File name: {source["file_name"]}

Source text:
<<<SOURCE_TEXT_START
{source["source_text"]}
SOURCE_TEXT_END>>>
"""


def get_response_text(response) -> str:
    """
    Robustly extract text from a google-genai response.
    response.text is usually enough, but this also checks candidate parts.
    """
    text = getattr(response, "text", None)
    if text:
        return text.strip()

    chunks = []
    try:
        for cand in response.candidates or []:
            content = getattr(cand, "content", None)
            parts = getattr(content, "parts", None) if content else None
            if not parts:
                continue
            for part in parts:
                part_text = getattr(part, "text", None)
                if part_text:
                    chunks.append(part_text)
    except Exception:
        pass

    return "\n".join(chunks).strip()


def summarize_empty_response(response) -> str:
    """
    Helpful debug string when Gemini returns no text.
    """
    bits = []

    try:
        for i, cand in enumerate(response.candidates or []):
            finish_reason = getattr(cand, "finish_reason", None)
            safety_ratings = getattr(cand, "safety_ratings", None)
            bits.append(f"candidate_{i}_finish_reason={finish_reason}")
            if safety_ratings:
                bits.append(f"candidate_{i}_safety_ratings={safety_ratings}")
    except Exception as e:
        bits.append(f"could_not_read_candidates={repr(e)}")

    try:
        usage = getattr(response, "usage_metadata", None)
        if usage:
            bits.append(f"usage_metadata={usage}")
    except Exception:
        pass

    return " | ".join(bits) if bits else "No response diagnostics available."


def parse_stage1_text(raw: str) -> dict:
    """
    Parse plain-text Stage 1 response.
    Expected:
    DECISION: PROCESS
    CONFIDENCE: 0.95
    REASON: ...
    """
    text = (raw or "").strip()

    decision_match = re.search(r"\b(PROCESS|SKIP)\b", text, flags=re.IGNORECASE)
    if not decision_match:
        raise ValueError(f"Could not find PROCESS/SKIP in model output: {text[:500]!r}")

    decision = decision_match.group(1).upper()

    conf_match = re.search(
        r"CONFIDENCE\s*:\s*([01](?:\.\d+)?)",
        text,
        flags=re.IGNORECASE,
    )
    confidence = float(conf_match.group(1)) if conf_match else 0.5
    confidence = max(0.0, min(1.0, confidence))

    reason_match = re.search(
        r"REASON\s*:\s*(.+)",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )
    reason = reason_match.group(1).strip() if reason_match else "Stage 1 decision parsed from plain-text model output."

    # Keep reason short and single-line.
    reason = re.sub(r"\s+", " ", reason).strip()
    reason = reason[:500]

    return {
        "stage1_decision": decision,
        "stage1_confidence": confidence,
        "stage1_reason": reason,
    }


def validate_stage1_output(obj: dict, source: dict) -> list:
    errors = []

    if not isinstance(obj, dict):
        return ["Output is not a dictionary"]

    decision = obj.get("stage1_decision")
    if decision not in ALLOWED_STAGE1_DECISIONS:
        errors.append(f"stage1_decision must be PROCESS or SKIP, got {decision!r}")

    conf = obj.get("stage1_confidence")
    if not isinstance(conf, (int, float)):
        errors.append("stage1_confidence must be numeric")
    elif not (0.0 <= float(conf) <= 1.0):
        errors.append("stage1_confidence must be between 0.0 and 1.0")

    reason = obj.get("stage1_reason")
    if not isinstance(reason, str) or not reason.strip():
        errors.append("stage1_reason must be a non-empty string")

    return errors


def normalize_stage1_output(obj: dict, source: dict) -> dict:
    """
    Python adds all identity fields.
    The model never copies URLs/file names/IDs.
    """
    return {
        "id": str(source["id"]),
        "claim": source.get("claim", ""),
        "url": source["url"],
        "file_name": source["file_name"],
        "stage1_decision": obj["stage1_decision"],
        "stage1_confidence": float(obj["stage1_confidence"]),
        "stage1_reason": obj["stage1_reason"].strip(),
        "source_text_original_chars": source.get("source_text_original_chars", 0),
        "source_text_packed_chars": source.get("source_text_packed_chars", 0),
        "source_text_was_packed": source.get("source_text_was_packed", False),
        "load_error": source.get("load_error"),
        "stage": "stage1_api_triage",
        "model": STAGE1_MODEL,
        "run_timestamp_utc": utc_now(),
    }


## 6. Stage 1 API call with retries

The fallback is intentionally **fail-open**: if the API repeatedly fails but a source record exists, it returns `PROCESS` with low confidence so later Stage 2 can make the relevance decision. This avoids silently losing potentially useful evidence.


In [11]:
def call_stage1_api(source: dict) -> dict:
    prompt = build_stage1_prompt(source)
    last_error = None
    raw_text = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.models.generate_content(
                model=STAGE1_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=STAGE1_TEMPERATURE,
                    max_output_tokens=512,
                    thinking_config=types.ThinkingConfig(thinking_budget=0),
                ),
            )

            raw_text = get_response_text(response)

            if not raw_text:
                diagnostics = summarize_empty_response(response)
                raise ValueError(f"Gemini returned empty text. Diagnostics: {diagnostics}")

            parsed = parse_stage1_text(raw_text)
            errors = validate_stage1_output(parsed, source)

            if errors:
                raise ValueError("; ".join(errors))

            return normalize_stage1_output(parsed, source)

        except Exception as e:
            last_error = repr(e)
            if "429" in last_error or "RESOURCE_EXHAUSTED" in last_error:
                sleep_s = min(20 * attempt, 90) + random.random() * 5
            else:
                sleep_s = min(2 ** attempt, 20) + random.random()

            print(
                f"Stage 1 attempt {attempt}/{MAX_RETRIES} failed for "
                f"{source['id']} {source['file_name']}: {last_error}. Sleeping {sleep_s:.1f}s"
            )
            time.sleep(sleep_s)

    # Stage 1 fail-open fallback:
    # Safer to PROCESS than to accidentally lose evidence.
    return {
        "id": str(source["id"]),
        "claim": source.get("claim", ""),
        "url": source["url"],
        "file_name": source["file_name"],
        "stage1_decision": "PROCESS",
        "stage1_confidence": 0.0,
        "stage1_reason": f"Fail-open fallback after Stage 1 API/validation failure: {last_error}",
        "source_text_original_chars": source.get("source_text_original_chars", 0),
        "source_text_packed_chars": source.get("source_text_packed_chars", 0),
        "source_text_was_packed": source.get("source_text_was_packed", False),
        "load_error": source.get("load_error"),
        "stage": "stage1_api_triage",
        "model": STAGE1_MODEL,
        "run_timestamp_utc": utc_now(),
        "raw_response_preview": (raw_text or "")[:1000],
    }

In [12]:
STAGE1_BATCH_RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "results": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "item_index": {"type": "integer"},
                    "decision": {"type": "string", "enum": ["PROCESS", "SKIP"]},
                    "confidence": {"type": "number"},
                    "reason": {"type": "string"},
                },
                "required": ["item_index", "decision", "confidence", "reason"],
            },
        }
    },
    "required": ["results"],
}


def build_stage1_batch_prompt(batch):
    packed_items = []

    for i, source in enumerate(batch):
        packed_items.append({
            "item_index": i,
            "id": str(source["id"]),
            "claim": source.get("claim", ""),
            "url": source["url"],
            "file_name": source["file_name"],
            "source_text": source.get("source_text", ""),
        })

    expected_indexes = list(range(len(batch)))

    return f"""You are checking whether scraped sources contain usable readable text.

You are NOT judging whether each source is relevant to the claim.
You are NOT checking whether any claim is true.
You are only deciding whether each source has meaningful readable content that a later stage can inspect.

Allowed decisions:
PROCESS = readable content exists.
SKIP = no usable readable content exists.

Choose PROCESS if the source contains any meaningful article text, post text, archive text, screenshot description, claim text, or useful page content.

Choose SKIP only if the source is empty, near-empty, login-only, access-denied, JavaScript-error-only, or menu/footer/cookie-shell-only.

Critical output rules:
- Return exactly {len(batch)} results.
- Required item_index values are exactly: {expected_indexes}
- Do not omit any item_index.
- Do not duplicate any item_index.
- Do not copy source text into the reason.
- Keep reasons short.
- Do not decide whether the source proves, supports, or refutes the claim.

Return strict JSON only:
{{
  "results": [
    {{
      "item_index": 0,
      "decision": "PROCESS",
      "confidence": 0.95,
      "reason": "readable article/post/archive/page content exists"
    }}
  ]
}}

Input batch:
{json.dumps(packed_items, ensure_ascii=False)}
"""


def make_stage1_batch_config():
    return types.GenerateContentConfig(
        temperature=STAGE1_TEMPERATURE,
        response_mime_type="application/json",
        response_schema=STAGE1_BATCH_RESPONSE_SCHEMA,
        max_output_tokens=4096,
        thinking_config=types.ThinkingConfig(thinking_budget=0),
    )


def normalize_stage1_batch_item(model_item, source):
    decision = str(model_item.get("decision", "")).upper().strip()
    if decision not in ALLOWED_STAGE1_DECISIONS:
        decision = "PROCESS"

    try:
        confidence = float(model_item.get("confidence", 0.5))
    except Exception:
        confidence = 0.5

    confidence = max(0.0, min(1.0, confidence))
    reason = re.sub(r"\s+", " ", str(model_item.get("reason", "") or "")).strip()
    if not reason:
        reason = "Stage 1 batch API triage decision."

    return {
        "id": str(source["id"]),
        "claim": source.get("claim", ""),
        "url": source["url"],
        "file_name": source["file_name"],
        "stage1_decision": decision,
        "stage1_confidence": confidence,
        "stage1_reason": reason[:500],
        "source_text_original_chars": source.get("source_text_original_chars", 0),
        "source_text_packed_chars": source.get("source_text_packed_chars", 0),
        "source_text_was_packed": source.get("source_text_was_packed", False),
        "load_error": source.get("load_error"),
        "stage": "stage1_api_triage_batched",
        "model": STAGE1_MODEL,
        "run_timestamp_utc": utc_now(),
    }


def extract_stage1_batch_json(response):
    parsed = getattr(response, "parsed", None)
    if parsed:
        if isinstance(parsed, dict):
            return parsed
        try:
            return json.loads(json.dumps(parsed))
        except Exception:
            pass

    raw = get_response_text(response)
    if not raw:
        diagnostics = summarize_empty_response(response)
        raise ValueError(f"Empty batch response. Diagnostics: {diagnostics}")

    raw = re.sub(r"^```(?:json)?\s*", "", raw.strip(), flags=re.IGNORECASE)
    raw = re.sub(r"\s*```$", "", raw).strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        start = raw.find("{")
        end = raw.rfind("}")
        if start >= 0 and end > start:
            return json.loads(raw[start:end + 1])
        raise


def call_stage1_batch_api(batch):
    prompt = build_stage1_batch_prompt(batch)
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.models.generate_content(
                model=STAGE1_MODEL,
                contents=prompt,
                config=make_stage1_batch_config(),
            )

            parsed = extract_stage1_batch_json(response)
            results = parsed.get("results", [])

            if not isinstance(results, list):
                raise ValueError("Batch response results is not a list")

            by_index = {}
            for item in results:
                if not isinstance(item, dict):
                    continue
                try:
                    idx = int(item.get("item_index"))
                except Exception:
                    continue
                if 0 <= idx < len(batch):
                    by_index[idx] = item

            outputs_by_index = {}

            # Keep all valid returned items.
            for i, source in enumerate(batch):
                model_item = by_index.get(i)
                if model_item is None:
                    continue

                out = normalize_stage1_batch_item(model_item, source)
                errors = validate_stage1_output(out, source)

                if not errors:
                    outputs_by_index[i] = out

            missing = [i for i in range(len(batch)) if i not in outputs_by_index]

            # If only a few items are missing, do NOT retry the whole batch.
            # Use the original API triage for only those missing items.
            if missing:
                print(f"Batch returned {len(outputs_by_index)}/{len(batch)} items; filling missing indexes with per-source API: {missing}")
                for i in missing:
                    outputs_by_index[i] = call_stage1_api(batch[i])

            return [outputs_by_index[i] for i in range(len(batch))]

        except Exception as e:
            last_error = repr(e)

            # 429 needs longer backoff than normal validation errors.
            if "429" in last_error or "RESOURCE_EXHAUSTED" in last_error:
                sleep_s = min(20 * attempt, 90) + random.random() * 5
            else:
                sleep_s = min(2 ** attempt, 20) + random.random()

            print(
                f"Stage 1 batch attempt {attempt}/{MAX_RETRIES} failed for "
                f"batch size {len(batch)}: {last_error}. Sleeping {sleep_s:.1f}s"
            )
            time.sleep(sleep_s)

    # API fallback, not deterministic fallback:
    # If the whole batch repeatedly fails, split into per-source API calls.
    print("Batch failed after retries; falling back to per-source API calls for this batch.")
    return [call_stage1_api(source) for source in batch]


def make_batches(items, batch_size):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]


## 7. Resume helpers

The output is a JSON list keyed by:

```text
claim_id || file_name || source_url
```

Already processed source records are skipped on rerun.


In [13]:

def load_json_list_if_exists(path: Path) -> list:
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise TypeError(f"Expected JSON list at {path}, got {type(data)}")
    return data


def save_json_atomic(data, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    with tmp_path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    shutil.move(str(tmp_path), str(path))


def index_stage1_outputs(outputs: list) -> dict:
    indexed = {}
    for item in outputs:
        if not isinstance(item, dict):
            continue
        if {"id", "file_name", "url"}.issubset(item.keys()):
            indexed[f'{item["id"]}||{item["file_name"]}||{item["url"]}'] = item
    return indexed

existing_outputs = load_json_list_if_exists(STAGE1_OUT)
existing_by_key = index_stage1_outputs(existing_outputs)

pending_sources = [s for s in source_records if source_key(s) not in existing_by_key]

print("Existing Stage 1 outputs:", len(existing_by_key))
print("Pending Stage 1 sources:", len(pending_sources))


Existing Stage 1 outputs: 0
Pending Stage 1 sources: 8979



## 8. Run Stage 1

This cell saves after every completed API result, so it is safe to interrupt and resume.


In [14]:
all_outputs_by_key = dict(existing_by_key)

pending_batches = list(make_batches(pending_sources, STAGE1_BATCH_SIZE))

print("Pending Stage 1 sources:", len(pending_sources))
print("Pending Stage 1 API batches:", len(pending_batches))
print("Batch size:", STAGE1_BATCH_SIZE)
print("MAX_WORKERS:", MAX_WORKERS)

completed_batches = 0

if not pending_sources:
    print("Nothing to do. Stage 1 output already exists for all source records.")
else:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_batch = {
            executor.submit(call_stage1_batch_api, batch): batch
            for batch in pending_batches
        }

        for future in tqdm(as_completed(future_to_batch), total=len(future_to_batch), desc="Stage 1 batched API triage"):
            batch = future_to_batch[future]

            try:
                results = future.result()
            except Exception as e:
                print("Unexpected executor-level batch failure; falling back to per-source API for this batch:", repr(e))
                results = [call_stage1_api(source) for source in batch]

            for source, result in zip(batch, results):
                key = source_key(source)
                all_outputs_by_key[key] = result

            completed_batches += 1

            if completed_batches % SAVE_AFTER_EVERY_COMPLETED_BATCHES == 0:
                save_json_atomic(list(all_outputs_by_key.values()), STAGE1_OUT)
                print(
                    f"Saved {len(all_outputs_by_key)}/{len(source_records)} outputs "
                    f"after {completed_batches}/{len(pending_batches)} completed batches"
                )

stage1_outputs = list(all_outputs_by_key.values())
save_json_atomic(stage1_outputs, STAGE1_OUT)

print("Saved Stage 1 outputs:", STAGE1_OUT)
print("Total Stage 1 outputs:", len(stage1_outputs))

Pending Stage 1 sources: 8979
Pending Stage 1 API batches: 2245
Batch size: 4
MAX_WORKERS: 3


Stage 1 batched API triage:   0%|          | 0/2245 [00:00<?, ?it/s]

Saved 40/8979 outputs after 10/2245 completed batches
Saved 80/8979 outputs after 20/2245 completed batches
Saved 120/8979 outputs after 30/2245 completed batches
Saved 160/8979 outputs after 40/2245 completed batches
Stage 1 batch attempt 1/5 failed for batch size 4: ClientError("429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}"). Sleeping 21.4s
Saved 200/8979 outputs after 50/2245 completed batches
Saved 240/8979 outputs after 60/2245 completed batches
Saved 280/8979 outputs after 70/2245 completed batches
Saved 320/8979 outputs after 80/2245 completed batches
Saved 360/8979 outputs after 90/2245 completed batches
Stage 1 batch attempt 1/5 failed for batch size 4: ClientError("429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to


## 9. Inspect Stage 1 decisions

Look for obvious failures:

- Empty/login/error pages should usually be `SKIP`.
- Perma.cc records, screenshots with meaningful text, readable posts, and article pages should usually be `PROCESS`.
- Stage 1 should not say whether a source is relevant to the claim.


In [15]:
df_stage1 = pd.DataFrame(stage1_outputs)

# Keep a useful display order.
cols = [
    "id",
    "file_name",
    "stage1_decision",
    "stage1_confidence",
    "stage1_reason",
    "source_text_original_chars",
    "source_text_packed_chars",
    "url",
]
cols = [c for c in cols if c in df_stage1.columns]

display(df_stage1[cols].sort_values(["id", "file_name"]).reset_index(drop=True))

print("Decision counts:")
print(df_stage1["stage1_decision"].value_counts(dropna=False))

print("\nPer-claim source counts:")
print(pd.crosstab(df_stage1["id"], df_stage1["stage1_decision"]))


,id,file_name,stage1_decision,stage1_confidence,stage1_reason,source_text_original_chars,source_text_packed_chars,url
0,0_ambigsnopes,1023.txt,SKIP,0.95,empty source text,0,0,https://vinguptamd.com
1,0_ambigsnopes,177.txt,SKIP,0.95,empty source text,0,0,https://www.nbcnews.com/politics/donald-trump/...
2,0_ambigsnopes,317.txt,SKIP,0.95,empty source text,0,0,https://www.bloomberg.com/news/articles/2020-0...
3,0_ambigsnopes,401.txt,SKIP,0.95,empty source text,0,0,https://www.cnn.com/2024/06/27/politics/read-b...
4,0_ambigsnopes,405.txt,SKIP,0.95,empty source text,0,0,https://www.reckitt.com/media-landing/press-re...
...,...,...,...,...,...,...,...,...
8974,9_ambigsnopes,66.txt,SKIP,0.95,empty source text,0,0,https://support.apple.com/en-au/HT202743
8975,9_ambigsnopes,77.txt,SKIP,0.95,empty source text,0,0,https://www.gov.uk/government/news/launch-of-l...
8976,9_ambigsnopes,787.txt,SKIP,0.95,empty source text,0,0,https://www.gov.uk/alerts
8977,9_ambigsnopes,834.txt,SKIP,0.95,empty source text,0,0,https://www.bbc.com/news/uk-64999417


Decision counts:
stage1_decision
PROCESS    5741
SKIP       3238
Name: count, dtype: int64

Per-claim source counts:
stage1_decision  PROCESS  SKIP
id                            
0_ambigsnopes          0    10
100_ambigsnopes        0    10
104_ambigsnopes        0    10
106_ambigsnopes        0     8
107_ambigsnopes        0     8
...                  ...   ...
95_ambigsnopes         8     1
96_ambigsnopes         0     1
98_ambigsnopes         3     0
99_ambigsnopes         0     4
9_ambigsnopes          0     6

[1158 rows x 2 columns]



## 10. Final Stage 1 validation summary

This re-runs the deterministic validator over saved outputs and reports any identity/schema issues.


In [16]:

source_by_key = {source_key(s): s for s in source_records}
validation_rows = []

for out in stage1_outputs:
    key = f'{out.get("id")}||{out.get("file_name")}||{out.get("url")}'
    source = source_by_key.get(key)
    if source is None:
        validation_rows.append({
            "key": key,
            "valid": False,
            "errors": "Output key not found in Stage 0 source records",
        })
        continue
    errors = validate_stage1_output(out, source)
    validation_rows.append({
        "key": key,
        "id": out.get("id"),
        "file_name": out.get("file_name"),
        "valid": not errors,
        "errors": "; ".join(errors),
    })

validation_df = pd.DataFrame(validation_rows)
display(validation_df)

if validation_df["valid"].all():
    print("✅ All saved Stage 1 outputs passed schema and identity validation.")
else:
    print("⚠️ Some Stage 1 outputs failed validation. Inspect the table above.")


,key,id,file_name,valid,errors
0,30354_exclaim||30354_5.json||https://money.cnn...,30354_exclaim,30354_5.json,True,
1,30354_exclaim||30354_6.json||https://abcnews.g...,30354_exclaim,30354_6.json,True,
2,30354_exclaim||30354_7.json||https://www.forbe...,30354_exclaim,30354_7.json,True,
3,30354_exclaim||30354_8.json||https://www.wsj.c...,30354_exclaim,30354_8.json,True,
4,30354_exclaim||30354_9.json||https://www.theat...,30354_exclaim,30354_9.json,True,
...,...,...,...,...,...
8974,218_ambigsnopes||900.txt||https://www.newspape...,218_ambigsnopes,900.txt,True,
8975,218_ambigsnopes||895.txt||https://www.newspape...,218_ambigsnopes,895.txt,True,
8976,218_ambigsnopes||1099.txt||https://www.newspap...,218_ambigsnopes,1099.txt,True,
8977,218_ambigsnopes||649.txt||https://www.npr.org/...,218_ambigsnopes,649.txt,True,


✅ All saved Stage 1 outputs passed schema and identity validation.



## 11. Next stage handoff

Stage 2 should read:

```python
/content/outputs/stage0/flattened_sources_TARGET10.json
/content/outputs/stage1/stage1_api_triage_TARGET10.json
```

Then it should run claim-aware evidence extraction only for source records where:

```python
stage1_decision == "PROCESS"
```

Remember: Stage 2 should **not** receive `original_rating`.


In [17]:
from pathlib import Path
import shutil

# Drive output folder
DRIVE_STAGE1_DIR = Path("/content/drive/MyDrive/CheckThat_Task3_Dataset/outputs/stage1")
DRIVE_STAGE1_DIR.mkdir(parents=True, exist_ok=True)

# Copy Stage 1 output from Colab runtime to Drive
drive_stage1_path = DRIVE_STAGE1_DIR / STAGE1_OUT.name
shutil.copy2(STAGE1_OUT, drive_stage1_path)

print("Saved Stage 1 output to Drive:")
print(drive_stage1_path)

Saved Stage 1 output to Drive:
/content/drive/MyDrive/CheckThat_Task3_Dataset/outputs/stage1/stage1_api_triage_test_dataset.json


In [18]:
from pathlib import Path
import shutil

# Source file in Colab runtime
stage0_runtime_path = STAGE0_OUT

# Destination folder in Drive
drive_stage0_dir = Path("/content/drive/MyDrive/CheckThat_Task3_Dataset/outputs/stage0")
drive_stage0_dir.mkdir(parents=True, exist_ok=True)

# Destination file
drive_stage0_path = drive_stage0_dir / stage0_runtime_path.name

# Copy
shutil.copy2(stage0_runtime_path, drive_stage0_path)

print("Saved Stage 0 flattened sources to Drive:")
print(drive_stage0_path)

Saved Stage 0 flattened sources to Drive:
/content/drive/MyDrive/CheckThat_Task3_Dataset/outputs/stage0/flattened_sources_test_dataset.json
